# Спектральное представление: собери поле из узоров
По мотивам приложенного слайда. Здесь **искусственное скалярное поле**, а не реальные температурные наблюдения. Единицы условные.

Представьте музыкальный аккорд: можно записать колебание во времени, а можно перечислить ноты и их громкости. Для поля на сфере «ноты» — сферические гармоники, «громкости» — коэффициенты (они могут быть и отрицательными).

**Как запустить:** откройте ноутбук в JupyterLab или Google Colab и выполните ячейки сверху вниз. Первая ячейка устанавливает библиотеки; нужен интернет. Ползунки пересчитывают рисунки. Если элементы управления не появились, обычные вызовы функций с числами тоже работают.

**Проверка:** кодовые ячейки (кроме установки зависимостей) и все встроенные `assert` выполнены локально без графического интерфейса. Максимальная ошибка восстановления коэффициентов — около 5.35×10⁻¹⁵, поля на сетке — 1.19×10⁻¹⁴. Работа ползунков в браузере и Colab отдельно не проверена.



In [ ]:
%pip install "numpy>=1.24" "scipy>=1.15" "matplotlib>=3.7" "ipywidgets>=8"

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y
import ipywidgets as widgets
from IPython.display import display
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': False})

## 1. Сначала обычные волны
Возьмём периодическое поле $f(x)=\sin x+0.5\cos(3x)+0.25\sin(8x)$.

Слева — три готовых узора; справа — их сумма. Коэффициент меняет вклад узора, а не его форму. Ограничение $K$ оставляет только частоты не выше $K$.

**Попробуйте:** при $K=1$ меняйте коэффициент третьей волны. Исходное поле изменится, а усечённое — нет. Затем поставьте $K=8$.

In [ ]:
def fourier(K=3, a1=1., a3=.5, a8=.25):
    x=np.linspace(0, 2*np.pi, 600)
    waves=[a1*np.sin(x), a3*np.cos(3*x), a8*np.sin(8*x)]
    target=sum(waves)
    approx=sum((w if k<=K else np.zeros_like(x)) for k,w in zip([1,3,8],waves))
    fig,ax=plt.subplots(1,2,figsize=(11,3.5))
    for w,label in zip(waves,['a₁ sin(x)','a₃ cos(3x)','a₈ sin(8x)']): ax[0].plot(x,w,label=label)
    ax[1].plot(x,target,color='0.5',label='Полное поле')
    ax[1].plot(x,approx,'--',label=f'Усечение K={K}')
    for a in ax:
        a.set(xlabel='x, радианы',ylabel='Значение поля',ylim=(-3.1,3.1))
        a.legend()
    plt.tight_layout(); plt.show()
    
widgets.interact(fourier, K=widgets.IntSlider(value=3,min=0,max=8),
                 a1=widgets.FloatSlider(value=1,min=-1,max=1,step=.1),
                 a3=widgets.FloatSlider(value=.5,min=-1,max=1,step=.1),
                 a8=widgets.FloatSlider(value=.25,min=-1,max=1,step=.05));

interactive(children=(IntSlider(value=3, description='K', max=8), FloatSlider(value=1.0, description='a1', max…

## 2.  Сфеерические гармоники. На сфере волны становятся узорами
На слайде $\lambda$ — долгота, $\varphi$ — широта. 
В SciPy аргументы другие: $\theta=\pi/2-\varphi$ — угол от северного полюса, второй угловой аргумент — долгота.

Используем **вещественный ортонормированный базис** $R_{\ell m}$, составленный из действительных и мнимых частей комплексных $Y_{\ell m}$:

$$R_{\ell 0}=Y_{\ell 0},\qquad R_{\ell m}=\sqrt2\,\operatorname{Re}Y_{\ell m}\ (m>0),\qquad R_{\ell m}=\sqrt2\,\operatorname{Im}Y_{\ell,|m|}\ (m<0).$$

В этой договорённости знак $m$ выбирает косинусную или синусную часть. Коэффициенты в других соглашениях могут отличаться знаками.

- $\ell=0$ — постоянный фон; $\ell=1$ — диполь; рост $\ell$ позволяет описывать более мелкие структуры.
- $|m|$ задаёт число периодов по долготе; $m=0$ — нет зависимости от долготы.
- Для каждой степени допустимо $-\ell\le m\le\ell$: всего $2\ell+1$ узоров.

**Попробуйте:** $(\ell,m)=(0,0),(1,0),(2,0),(2,2),(5,3),(5,-3)$. Шар не меняет форму: цвет показывает значение поля на его поверхности. Прямоугольник — развёртка по широте и долготе, которая растягивает полярные области.

In [ ]:
def real_Y(ell,m,lat,lon):
    y=sph_harm_y(ell,abs(m),np.pi/2-lat,lon)
    return y.real if m==0 else np.sqrt(2)*(y.real if m>0 else y.imag)

lon=np.linspace(-np.pi,np.pi,181)
lat=np.linspace(-np.pi/2,np.pi/2,91)
LON,LAT=np.meshgrid(lon,lat)
X=np.cos(LAT)*np.cos(LON); Y=np.cos(LAT)*np.sin(LON); Z=np.sin(LAT)

def harmonic(ell=2,m=1,azimuth=30):
    field=real_Y(ell,m,LAT,LON)
    vmax=max(np.abs(field).max(),1e-10)
    norm=plt.Normalize(-vmax,vmax)
    fig=plt.figure(figsize=(11,4))
    a=fig.add_subplot(121,projection='3d')
    a.plot_surface(X,Y,Z,facecolors=plt.cm.RdBu_r(norm(field)),
                   rstride=2,cstride=2,shade=False,linewidth=0)
    a.set_box_aspect((1,1,1)); a.view_init(elev=20,azim=azimuth)
    a.set_axis_off(); a.set_title(f'Один узор R({ell}, {m})')
    b=fig.add_subplot(122)
    im=b.pcolormesh(np.degrees(LON),np.degrees(LAT),field,
                   cmap='RdBu_r',norm=norm,shading='auto')
    b.set(xlabel='Долгота, °',ylabel='Широта, °',yticks=[-90,-45,0,45,90])
    fig.colorbar(im,ax=b,label='Значение R (шкала для выбранного узора)')
    plt.tight_layout(); plt.show()

ell_slider=widgets.IntSlider(value=2,min=0,max=8,description='ℓ')
m_slider=widgets.IntSlider(value=1,min=-2,max=2,description='m')
rot_slider=widgets.IntSlider(value=30,min=-180,max=180,description='Поворот')

def adjust_m(change):
    # Updating bounds also clamps m to the valid interval.
    m_slider.min=-change['new']; m_slider.max=change['new']
    
ell_slider.observe(adjust_m,names='value')
display(widgets.VBox([ell_slider,m_slider,rot_slider]),
        widgets.interactive_output(harmonic,dict(ell=ell_slider,m=m_slider,azimuth=rot_slider)))

Output()

Дальше идея такая: сложную карту на сфере можно собрать, складывая эти узоры с разными весами.

Представь, что у нас есть карта температуры: в каждой точке сетки на сфере записано значение. Это поле T_grid.

Хотим описать ту же карту другим способом: не «какая температура в каждой точке», а «сколько какого узора нужно сложить, чтобы получить эту карту».


В ноутбуке для проверки мы сами создаём карту по известному рецепту:
$$
T = 0.5 R_{0,0} + 1.4 R_{1,0} + 1.0 R_{2,2} + 0.8 R_{5,-3} + 0.55 R_{8,6}.
$$

Здесь $R_{\ell,m}$ — готовый узор из шага 2, а число перед ним — его вес. Сложение происходит в каждой точке: берём значения пяти узоров в этой точке, умножаем на веса и складываем.


## 3. Сетка → коэффициенты → поле
Теперь начнём со значений на сетке. Исходное поле — сумма пяти узоров. Мы знаем её рецепт и можем проверить, восстановит ли преобразование веса.

Для вещественного ортонормированного базиса:
$$a_{\ell m}=\int_{-\pi}^{\pi}\int_{-\pi/2}^{\pi/2}T(\lambda,\varphi)R_{\ell m}(\varphi,\lambda)\cos\varphi\,d\varphi\,d\lambda.$$

Это взвешенное «сходство» поля с узором. Обычное среднее по равномерным широтам неверно: площадь ячеек у полюсов меньше. Ниже интеграл вычисляется квадратурой Гаусса по $\mu=\sin\varphi$ и равномерной сеткой по долготе. Это учебная квадратурная сетка, не требование к любой модели.

In [6]:
MAX_L=8
mu, wg=np.polynomial.legendre.leggauss(24)
qlon=np.linspace(-np.pi,np.pi,64,endpoint=False)
QLON, QLAT=np.meshgrid(qlon,np.arcsin(mu))
weights=wg[:,None]*np.ones((1,len(qlon)))*(2*np.pi/len(qlon))
modes=[(ell,m) for ell in range(MAX_L+1) for m in range(-ell,ell+1)]
basis_q=np.array([real_Y(ell,m,QLAT,QLON) for ell,m in modes])
basis_plot=np.array([real_Y(ell,m,LAT,LON) for ell,m in modes])
recipe={(0,0):.5,(1,0):1.4,(2,2):1.,(5,-3):.8,(8,6):.55}
true_a=np.array([recipe.get(mode,0.) for mode in modes])
T_grid=np.einsum('k,kij->ij',true_a,basis_q)
# Прямое преобразование: интеграл приближён взвешенной суммой.
a=np.einsum('ij,kij,ij->k',T_grid,basis_q,weights)
# Обратное преобразование: сумма узоров с найденными весами.
T_back=np.einsum('k,kij->ij',a,basis_q)
print('Максимальная ошибка коэффициентов:',np.max(np.abs(a-true_a)))
print('Максимальная ошибка на сетке:',np.max(np.abs(T_back-T_grid)))
for mode,coef in zip(modes,a):
    if abs(coef)>1e-10: print(f'(ℓ,m)={mode}: a={coef:.3f}')
# Проверяем ортонормированность всех 81 функций и обратимость примера.
G=np.einsum('kij,nij,ij->kn',basis_q,basis_q,weights)
assert np.allclose(G,np.eye(len(modes)),atol=1e-11)
assert np.allclose(a,true_a,atol=1e-11)
assert np.allclose(T_back,T_grid,atol=1e-11)

Максимальная ошибка коэффициентов: 5.346667724202034e-15
Максимальная ошибка на сетке: 1.1879386363489175e-14
(ℓ,m)=(0, 0): a=0.500
(ℓ,m)=(1, 0): a=1.400
(ℓ,m)=(2, 2): a=1.000
(ℓ,m)=(5, -3): a=0.800
(ℓ,m)=(8, 6): a=0.550


## 4. Что делает усечение L
Слева одно и то же исходное поле, справа — сумма только степеней $\ell\le L$. Цветовая шкала фиксирована. Число разрешённых коэффициентов равно $(L+1)^2$, хотя в этом примере ненулевых всего пять.

RMS — среднеквадратичная ошибка с учётом площади на сфере. Доля $\int T_L^2d\Omega/\int T^2d\Omega$ описывает сохранённый квадрат нормы поля; это не термодинамическая энергия.

**Попробуйте:** $L=0,1,2,4,5,8$. Почему между 2 и 4 ничего не меняется? Потому что в нашем рецепте нет степеней 3 и 4. При $L=8$ восстанавливается весь пример; произвольное поле может требовать гораздо большего $L$.

In [ ]:
T_full=np.einsum('k,kij->ij',a,basis_plot)
vmax=np.abs(T_full).max()
def truncate(L=2):
    keep=np.array([ell<=L for ell,m in modes])
    a_L=a*keep
    T_L=np.einsum('k,kij->ij',a_L,basis_plot)
    err=np.sqrt(np.sum((a-a_L)**2)/(4*np.pi))
    retained=np.sum(a_L**2)/np.sum(a**2)
    fig,axs=plt.subplots(1,2,figsize=(11,3.7),layout='constrained')
    for ax,F,title in zip(axs,[T_full,T_L],['Полное поле (L=8)',f'Усечение L={L}']):
        im=ax.pcolormesh(np.degrees(LON),np.degrees(LAT),F,
                        cmap='RdBu_r',vmin=-vmax,vmax=vmax,shading='auto')
        ax.set(title=title,xlabel='Долгота, °',ylabel='Широта, °')
    fig.colorbar(im,ax=axs,label='Поле, усл. ед.')
    plt.show()
    print(f'Разрешено {(L+1)**2} коэффициентов; RMS={err:.4f}; сохранено ∫T²dΩ: {retained:.1%}')
widgets.interact(truncate,L=widgets.IntSlider(value=2,min=0,max=MAX_L));

interactive(children=(IntSlider(value=2, description='L', max=8), Output()), _dom_classes=('widget-interact',)…

## 5. Зачем в псевдоспектральном методе оба представления?
Для линейных операторов коэффициенты часто удобны. Например, на сфере радиуса $r$:
$$\Delta R_{\ell m}=-\frac{\ell(\ell+1)}{r^2}R_{\ell m}.$$
Лапласиан поля можно получить простым умножением каждого коэффициента. А произведение двух полей удобно считать поточечно на сетке, после чего преобразовать результат обратно.

**Проверка идеи:** следующая ячейка вычисляет коэффициенты $T^2$ через сетку. У произведения возможны степени до $2L$. Поэтому используем более богатый базис и достаточно точную квадратуру, и лишь затем можем усечь результат. Без достаточного разрешения высокие частоты могут ошибочно попадать в низкие — это алиасинг.

In [ ]:
# У исходного T максимальная степень 8, у T² — не выше 16.
# 24 узла Гаусса и 64 долготы достаточны для этих проекций.
product_modes=[(ell,m) for ell in range(17) for m in range(-ell,ell+1)]
product_a=np.array([np.sum(T_grid**2*real_Y(ell,m,QLAT,QLON)*weights)
                    for ell,m in product_modes])
low=sum(c*c for c,(ell,m) in zip(product_a,product_modes) if ell<=8)
high=sum(c*c for c,(ell,m) in zip(product_a,product_modes) if ell>8)
print(f'Доля квадрата нормы T² в степенях выше 8: {high/(low+high):.2%}')
assert np.isclose(np.sum(product_a**2),np.sum(T_grid**4*weights),atol=1e-10)

## Проверьте себя
1. Если сменить знак одного коэффициента, что произойдёт? **Ответ:** вклад этого узора сменит знак; всё поле целиком сменит знак, только если сменить знаки всех коэффициентов.
2. Является ли $a_{00}$ средним поля? **Ответ:** с нашей нормировкой $R_{00}=1/\sqrt{4\pi}$, поэтому среднее равно $a_{00}/\sqrt{4\pi}$.
3. Можно ли при малом $L$ сохранить любые мелкие детали, увеличив число точек для рисунка? **Ответ:** нет. Частая сетка отрисовки не добавляет отсутствующих гармоник.
4. Выбирает ли спектральное представление физические уравнения и шаг по времени? **Ответ:** нет. Представление поля, уравнения и временная схема — отдельные решения, как и сказано на слайде.

### Источники и соглашения
- [SciPy: sph_harm_y](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.sph_harm_y.html) — порядок углов, нормировка, фаза Кондона—Шортли.
- [NIST DLMF §14.30](https://dlmf.nist.gov/14.30) — сферические гармоники и ортогональность.

Ноутбук опирается на присланный слайд; текст главы в запрос не приложен.